<a href="https://colab.research.google.com/github/bmulaw/412code/blob/master/ComputerVision_lab2_SAM_ObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Team Name/Member: Bernard MULAW and John BRITTO
#### Date: 09/11/2026

In [1]:
import platform
print(platform.python_version())

3.13.18


In [2]:
%pip -q install "opencv-python" "ultralytics" "segmentation-models-pytorch" "torch" "scikit-learn" "torchvision"

In [4]:
import torch
import segmentation_models_pytorch as smp
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from ultralytics import YOLO

# U-NET encoder
_ = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=1)

# DeepLab V3
_ = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)

# Mask R-CNN
_ = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)

# DETR
# _ = torch.hub.load("facebookreasearch/detr", "detr_resnet50", pretrained=True)

# YOLOv8n
_ = YOLO("yolov8n.pt")

print("All weights downloaded (U-Net, DeepLab v3, Mask R-CNN, DETR, YOLO")

All weights downloaded (U-Net, DeepLab v3, Mask R-CNN, DETR, YOLO


In [5]:
print(torch.__version__)
torch.cuda.is_available()

2.14.1+cpu


False

## Task 1.1: Implement Segmentation Metrics

In [42]:
import numpy as np

def iou_score(pred, true):
    """ pred: binary mask (H, W)
        true: binary mask (H, W)
        Return: IoU
    """
    # https://stackoverflow.com/questions/49338166/python-intersection-over-union
    pred = np.asarray(pred).astype(bool)
    true = np.asarray(true).astype(bool)
    intersection = pred * true # Logical AND
    union = pred + true # Logical OR

    IOU = intersection.sum()/float(union.sum())
    # intersection = np.intersect1d(pred, true)
    # union = len(pred | true)
    # print("union", union.sum())
    # print("intersection", intersection.sum())
    # print(IOU)
    return IOU


def dice_coefficient(pred, true) -> float:
    # Return: Dice coefficient
    numinator = 2 * len(pred & true)
    denominator = len(pred) + len(true)
    # print("numinator", numinator, "denominator", denominator)
    return numinator/ denominator


def pixel_accuracy(pred, true) -> float:
    # Return: Pixel accuracy
    correct = 0
    total = 0
    for i in range(len(true)):
        for j in range(len(true[i])):
            if pred[i][j] == true[i][j]: correct += 1
            total += 1
    return correct/total


def multi_class_iou(pred, true, num_classes):
    """
        pred: (H, W) long tensor, class label per pixel
        true: (H, W) long tensor
        Return: (ious , miou), where ious is an array of per-class IoU
        Note: if a class is absent in both pred and true, return NaN for that class
    """
    ious = []
    for cls in range(num_classes):
        pred_mask = (pred == cls)
        true_mask = (true == cls)

        intersection = (pred_mask * true_mask).sum().item()
        union = (pred_mask + true_mask).sum().item()

        ious.append(intersection / union) if union != 0 else ious.append(float('nan'))

    # Calculate mean IoU ignoring NaN values --> got from AI
    valid_ious = [iou for iou in ious if not torch.isnan(torch.tensor(iou))]
    miou = sum(valid_ious) / len(valid_ious) if len(valid_ious) > 0 else float('nan')

    return ious, miou

In [40]:
true_mask = np.array([[0 ,1 ,1] ,[0 ,1 ,0] ,[1 ,0 ,0]])
pred_mask = np.array([[0 ,1 ,1] ,[0 ,1 ,0] ,[1 ,0 ,0]])
assert iou_score(pred_mask , true_mask) == 1.0
assert dice_coefficient(pred_mask, true_mask) == 1.0

In [35]:
pred_mask = 1 - true_mask
# print(pred_mask, "\n\n", true_mask)
# print(iou_score(pred_mask, true_mask))
assert iou_score(pred_mask, true_mask) == 0.0

In [36]:
pred_mask = np.array([[1 ,1 ,0] ,[0 ,1 ,0] ,[1 ,0 ,0]])
print("IoU: ", iou_score(pred_mask, true_mask))
print("Dice: ", dice_coefficient(pred_mask, true_mask))
print("Pixel Acc:", pixel_accuracy(pred_mask, true_mask))

IoU:  0.6
Dice:  1.0
Pixel Acc: 0.7777777777777778


## Task 1.2: Imbalanced Data Trap

In [37]:
H, W = 100, 100
true_mask = np.zeros((H,W), dtype=np.uint8)
true_mask[40:60, 40:60] = 1  # 4% foreground

# Predictor: predict everything as background
pred_mask = np.zeros_like(true_mask)
print ("IoU: ", iou_score(pred_mask, true_mask))
print("Dice: ", dice_coefficient(pred_mask, true_mask))
print("Pixel Acc:", pixel_accuracy(pred_mask, true_mask))

IoU:  0.0
Dice:  1.0
Pixel Acc: 0.96


## Task 1.3: Multi-Class Segmentation Metrics

In [43]:
H, W = 100, 100
true_mask = np.zeros((100,100), dtype=np.uint8)
true_mask[20:50, 20:50] = 1
true_mask[60:90, 60:90] = 2

# Predictor: predict everything as background
pred_mask = np.zeros_like(true_mask)
pred_mask[25:55, 25:55] = 1
pred_mask[60:90, 60:90] = 2

ious, miou = multi_class_iou(pred_mask, true_mask, num_classes=3)
print("Per-class IoU: ", ious)
print("mIoU: ", miou)

Per-class IoU:  [0.9351032448377581, 0.5319148936170213, 1.0]
mIoU:  0.8223393794849265
